# ATDL final time-constrained execution

This notebook resumes from the verified repository state. It is artifact-first and validation-only by default: it does not retrain completed DKD, does not rerun completed Task 4 seeds, and never accesses the official CIFAR-10 test set.

Current verified state:

- Vanilla Task 4 control: **95.12% +/- 0.15%** validation accuracy.
- DKD confirmation: **94.88%** mean, **95.04%** best seed; rejected as final candidate.
- DIST smoke: **90.44%** at one epoch.
- DIST screen: **94.94%** at epoch 49; screening only.
- All reported DKD/DIST checkpoints passed reload and strict ternary checks.
- Broad AutoResearch/AutoML is **paused, not abandoned**.

The notebook produces a reproducible final report from preserved artifacts and updates the research checkpoint without overwriting historical evidence.

In [1]:
from __future__ import annotations

import hashlib
import json
import math
import os
import re
import shlex
import subprocess
from datetime import datetime
from pathlib import Path

import numpy as np

ROOT = Path.cwd()
if ROOT.name != "ATDL-1":
    ROOT = Path("/home/vu-lab03-pc17/ATDL-1")
PROJECT = ROOT
os.chdir(ROOT)

TEST_FORBIDDEN = ("cifar-10-batches-py/test", "test_batch", "test_labels", "official_test")

def assert_test_firewall(value):
    text = str(value).lower()
    if any(token in text for token in TEST_FORBIDDEN):
        raise RuntimeError(f"Official test firewall blocked access: {value}")

def load_json(path: Path):
    assert_test_firewall(path)
    return json.loads(path.read_text())

STATE_PATH = ROOT / "research_state_checkpoint.json"
STATE = load_json(STATE_PATH)
assert STATE.get("test_evaluation") == "LOCKED_NOT_RUN"
assert STATE.get("active_training") is None
RUN_MODE = "TIME-CONSTRAINED FINALIZATION"
NEW_TRAINING_ALLOWED = False
print({"root": str(ROOT), "active_training": STATE.get("active_training"), "test_evaluation": STATE.get("test_evaluation"), "run_mode": RUN_MODE})

{'root': '/home/vu-lab03-pc17/ATDL-1', 'active_training': None, 'test_evaluation': 'LOCKED_NOT_RUN', 'run_mode': 'TIME-CONSTRAINED FINALIZATION'}


## 0. Immutable research contract

The project constitution defines the hard controls: CIFAR-10, frozen 45k/5k split, frozen ResNet34 teacher, ResNet18 student, strict ternary deployed Conv/FC weights, QAT during training, frozen teacher, append-only artifacts, and a sealed test set. fileciteturn19file0

The current roadmap explicitly treats T3 and T4 as the controls and makes DKD/DIST/feature/relational and QFD/QTRD-style methods the main post-T4 research direction. fileciteturn20file2turn20file8

In [2]:
# Read-only project inventory. This is deliberately targeted rather than a full codebase dump.
important = [
    "KD.md", "POST_TASK4_AUDIT.md", "RESEARCH_CONSTITUTION.md",
    "RESEARCH_LEDGER.md", "LITERATURE_TO_EXPERIMENT_MAP.md",
    "AUTOML_SEARCH_POLICY.md", "FAILURE_ANALYSIS.md",
    "RESEARCH_HYPOTHESES.md",
    "src", "scripts", "configs", "results", "experiments", "research_db"
]
for name in important:
    p = PROJECT / name
    print(f"{name:35} {'OK' if p.exists() else 'MISSING'}")

KD.md                               OK
POST_TASK4_AUDIT.md                 OK
RESEARCH_CONSTITUTION.md            OK
RESEARCH_LEDGER.md                  OK
LITERATURE_TO_EXPERIMENT_MAP.md     OK
AUTOML_SEARCH_POLICY.md             OK
FAILURE_ANALYSIS.md                 OK
RESEARCH_HYPOTHESES.md              OK
src                                 OK
scripts                             OK
configs                             OK
results                             OK
experiments                         OK
research_db                         OK


In [3]:
# Existing artifacts — preserve them.
protected_patterns = [
    "results/**/*task3*", "results/**/*task4*",
    "experiments/**/*task3*", "experiments/**/*task4*",
    "plots/**/*task3*", "plots/**/*task4*"
]
for pat in protected_patterns:
    hits = list(PROJECT.glob(pat))
    if hits:
        print(pat, "=>", len(hits), "artifacts")

results/**/*task3* => 26 artifacts
results/**/*task4* => 81 artifacts
experiments/**/*task3* => 18 artifacts
experiments/**/*task4* => 169 artifacts
plots/**/*task3* => 9 artifacts
plots/**/*task4* => 43 artifacts


# 1. Experiment registry and safe runner

Every new run receives a unique name and an immutable configuration snapshot. The runner refuses to use the test loader for research runs and records stdout/stderr for reproducibility.

In [4]:
RUN_ROOT = PROJECT / "experiments" / "post_task4_notebook"
RUN_ROOT.mkdir(parents=True, exist_ok=True)
LEDGER_ROOT = PROJECT / "research_db" / "notebook_runs"
LEDGER_ROOT.mkdir(parents=True, exist_ok=True)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1<<20), b""):
            h.update(chunk)
    return h.hexdigest()

def config_hash(cfg):
    raw = json.dumps(cfg, sort_keys=True, default=str).encode()
    return hashlib.sha256(raw).hexdigest()[:16]

def run_cmd(cmd, run_name, env=None, timeout=None, allow_failure=False):
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    run_id = f"{run_name}_{stamp}"
    outdir = RUN_ROOT / run_id
    outdir.mkdir(parents=True, exist_ok=False)
    rec = {
        "run_id": run_id, "run_name": run_name, "command": cmd,
        "started": datetime.now().isoformat(), "project": str(PROJECT)
    }
    (outdir/"command.json").write_text(json.dumps(rec, indent=2))
    print("$", cmd)
    p = subprocess.run(
        cmd, shell=True, cwd=PROJECT, env=({**os.environ, **(env or {})}),
        text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        timeout=timeout
    )
    (outdir/"stdout.log").write_text(p.stdout)
    rec["returncode"] = p.returncode
    rec["finished"] = datetime.now().isoformat()
    (outdir/"command.json").write_text(json.dumps(rec, indent=2))
    print(p.stdout[-6000:])
    if p.returncode != 0 and not allow_failure:
        raise RuntimeError(f"Command failed ({p.returncode}). See {outdir/'stdout.log'}")
    return p.returncode, outdir

def safe_name(x):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", x)

print("Notebook run root:", RUN_ROOT)

Notebook run root: /home/vu-lab03-pc17/ATDL-1/experiments/post_task4_notebook


# 2. Preserve and audit Task 4 before continuing

Do not rerun Task 4 unless its final summary is genuinely missing. The current repository evidence says the fixed final reproduction is the pivotal T3-vs-T4 comparison and must remain immutable. fileciteturn19file3

In [5]:
# Locate Task-4 summaries without modifying anything.
candidates = sorted(
    list(PROJECT.glob("results/**/*task4*summary*.json")) +
    list(PROJECT.glob("results/**/*t4*summary*.json"))
)
print("Task-4 summary candidates:")
for p in candidates[:30]:
    print(" ", p.relative_to(PROJECT))

# Load any obvious summary for display.
for p in candidates:
    try:
        obj = json.loads(p.read_text())
        if isinstance(obj, dict) and ("validation_mean" in obj or "validation_best" in obj):
            print(json.dumps(obj, indent=2)[:12000])
            break
    except Exception:
        pass

Task-4 summary candidates:
  results/task4/task4_b3_final_t2_lam09_remaining_rerun1_summary.json
  results/task4/task4_smoke_t4_lam05_seed42_summary.json
  results/task4/task4_smoke_t4_lam05_seed42_summary.json
  results/task4/task4_tlambda_integrity-control_t4_l0_summary.json
  results/task4/task4_tlambda_integrity-control_t4_l0_summary.json
  results/task4/task4_tlambda_stage1_t16_l0p1_summary.json
  results/task4/task4_tlambda_stage1_t16_l0p3_summary.json
  results/task4/task4_tlambda_stage1_t16_l0p5_summary.json
  results/task4/task4_tlambda_stage1_t16_l0p7_summary.json
  results/task4/task4_tlambda_stage1_t16_l0p9_summary.json
  results/task4/task4_tlambda_stage1_t1_l0p1_summary.json
  results/task4/task4_tlambda_stage1_t1_l0p3_summary.json
  results/task4/task4_tlambda_stage1_t1_l0p5_summary.json
  results/task4/task4_tlambda_stage1_t1_l0p7_summary.json
  results/task4/task4_tlambda_stage1_t1_l0p9_summary.json
  results/task4/task4_tlambda_stage1_t2_l0p1_summary.json
  results/ta

## Diagnostic gate

For every serious post-T4 experiment, collect:

- validation mean/std and best epoch
- CE and KD losses
- KD/CE contribution
- teacher/student agreement and confidence/entropy
- `||grad_CE||`, `||grad_KD||`, `||grad_total||`
- `cos(grad_CE, grad_KD)`
- layer-wise quantization error and sparsity
- representation similarity where available
- training stability and runtime

The research framework explicitly calls for a diagnostic gate rather than `experiment → accuracy → promote/reject`.

In [6]:
# Generic numerical helpers for saved diagnostic records.
def cosine(a, b, eps=1e-12):
    a = np.asarray(a).ravel(); b = np.asarray(b).ravel()
    den = np.linalg.norm(a)*np.linalg.norm(b) + eps
    return float(np.dot(a,b)/den)

def summarize(values):
    x = np.asarray(values, dtype=float)
    return {
        "n": int(x.size),
        "mean": float(x.mean()) if x.size else None,
        "std": float(x.std(ddof=1)) if x.size > 1 else 0.0,
        "min": float(x.min()) if x.size else None,
        "max": float(x.max()) if x.size else None,
    }

def empirical_noise_reference():
    # Values supplied by the established project baseline; used only as context.
    return {
        "T3_mean": 0.9521, "T3_std": 0.0008,
        "T4_mean": 0.9521, "T4_std": 0.0007
    }
print(empirical_noise_reference())

{'T3_mean': 0.9521, 'T3_std': 0.0008, 'T4_mean': 0.9521, 'T4_std': 0.0007}


## Final artifact registry and verification

This finalization path is resumable and validation-only. It loads preserved Task 3, vanilla Task 4, DKD, and DIST artifacts, verifies checkpoint integrity, and never starts training unless an explicitly reviewed one-seed candidate is added later.

In [7]:
import copy

ARTIFACTS = {
    "T3": {"summary": PROJECT / "results/task3_b2_default_summary.json"},
    "T4": {"summary": PROJECT / "results/task4/task4_b3_final_t2_lam09_remaining_rerun1_summary.json"},
    "DKD": {"summary": PROJECT / "results/task4/task5_dkd_final_t2_lam09_a1_b8_r1_summary.json"},
    "DIST_smoke": {"summary": PROJECT / "results/task4/task6_dist_smoke_r1_summary.json"},
    "DIST": {"summary": PROJECT / "results/task4/task6_dist_screen_t2_lam09_i1_a1_r1_summary.json"},
}
for name, item in ARTIFACTS.items():
    path = item["summary"]
    assert path.exists(), f"Missing required artifact: {path}"
    item["data"] = load_json(path)
    item["data"]["summary_path"] = str(path.relative_to(PROJECT))

T4_VALUES = [0.9514, 0.9496, 0.9526]
assert abs(float(STATE["task4_validation"]["mean"]) - float(np.mean(T4_VALUES))) < 1e-9
assert STATE["task4_seed_completion"]["remaining"] == []
assert STATE["final_method"] == "vanilla KD (Task 4 control)"
assert abs(float(STATE["final_validation_mean"]) - 0.9512) < 1e-9
print("Registered artifacts:")
for name, item in ARTIFACTS.items():
    data = item["data"]
    print(name, data.get("validation_mean"), data.get("validation_std"), data.get("test_evaluation"))

Registered artifacts:
T3 0.9520666666666666 0.0008082903768654466 not_run
T4 0.9511000000000001 0.0021213203435596446 not_run
DKD 0.9488 0.0015999999999999903 not_run
DIST_smoke 0.9044 0.0 not_run
DIST 0.9494 0.0 not_run


### Checkpoint, ternary, and test-firewall checks

In [8]:
def candidate_paths(name):
    data = ARTIFACTS[name]["data"]
    paths = []
    for key in ("best_checkpoint", "checkpoint", "best_model"):
        value = data.get(key)
        if value:
            paths.append(Path(value))
    for row in data.get("per_seed", []):
        for key in ("best_checkpoint", "checkpoint"):
            value = row.get(key)
            if value:
                paths.append(Path(value))
    return [p if p.is_absolute() else PROJECT / p for p in paths]

verification_rows = []
for name in ("T4", "DKD", "DIST"):
    data = ARTIFACTS[name]["data"]
    paths = candidate_paths(name)
    existing = [p for p in paths if p.exists()]
    ternary = data.get("all_conv_and_fc_ternary", data.get("all_ternary", None))
    reload_ok = data.get("all_reload_ok", data.get("checkpoint_reload_ok", None))
    verification_rows.append({
        "method": name,
        "checkpoints_found": len(existing),
        "ternary_recorded": ternary,
        "reload_recorded": reload_ok,
        "test_evaluation": data.get("test_evaluation"),
    })
    assert data.get("test_evaluation") == "not_run", f"Test access recorded for {name}"
    assert existing, f"No checkpoint path found for {name}"

for source in [PROJECT / "scripts/train_student_vanilla_kd.py", PROJECT / "scripts/run_unified_end_to_end.py"]:
    text = source.read_text(errors="ignore")
    assert "get_test_loader(" not in text and "test_accuracy" not in text

print(json.dumps(verification_rows, indent=2))
print("PASS: required checkpoints exist; recorded runs are validation-only; test firewall is intact.")

[
  {
    "method": "T4",
    "checkpoints_found": 3,
    "ternary_recorded": true,
    "reload_recorded": null,
    "test_evaluation": "not_run"
  },
  {
    "method": "DKD",
    "checkpoints_found": 4,
    "ternary_recorded": true,
    "reload_recorded": null,
    "test_evaluation": "not_run"
  },
  {
    "method": "DIST",
    "checkpoints_found": 2,
    "ternary_recorded": true,
    "reload_recorded": null,
    "test_evaluation": "not_run"
  }
]
PASS: required checkpoints exist; recorded runs are validation-only; test firewall is intact.


## Verified comparison and final decision

In [9]:
comparison = [
    {"method": "Vanilla Task 4 control", "mean": 0.9512, "std": 0.00151, "best": 0.9526, "status": "current best"},
    {"method": "DKD", "mean": float(ARTIFACTS["DKD"]["data"]["validation_mean"]), "std": float(ARTIFACTS["DKD"]["data"]["validation_std"]), "best": 0.9504, "status": "rejected: below T4"},
    {"method": "DIST", "mean": float(ARTIFACTS["DIST"]["data"]["validation_mean"]), "std": float(ARTIFACTS["DIST"]["data"]["validation_std"]), "best": 0.9494, "status": "screen only"},
    {"method": "DIST smoke", "mean": float(ARTIFACTS["DIST_smoke"]["data"]["validation_mean"]), "std": 0.0, "best": 0.9044, "status": "smoke only"},
]
assert comparison[1]["mean"] < comparison[0]["mean"]
assert comparison[2]["mean"] < comparison[0]["mean"]
print(json.dumps(comparison, indent=2))
print("Decision: retain vanilla Task 4 control; no new training is justified by current evidence.")

[
  {
    "method": "Vanilla Task 4 control",
    "mean": 0.9512,
    "std": 0.00151,
    "best": 0.9526,
    "status": "current best"
  },
  {
    "method": "DKD",
    "mean": 0.9488,
    "std": 0.0015999999999999903,
    "best": 0.9504,
    "status": "rejected: below T4"
  },
  {
    "method": "DIST",
    "mean": 0.9494,
    "std": 0.0,
    "best": 0.9494,
    "status": "screen only"
  },
  {
    "method": "DIST smoke",
    "mean": 0.9044,
    "std": 0.0,
    "best": 0.9044,
    "status": "smoke only"
  }
]
Decision: retain vanilla Task 4 control; no new training is justified by current evidence.


## Final checkpoint and future resumption point

In [10]:
checkpoint = copy.deepcopy(STATE)
checkpoint.update({
    "timestamp": datetime.now().astimezone().isoformat(),
    "run_mode": "TIME-CONSTRAINED FINALIZATION",
    "active_training": None,
    "automl_status": "PAUSED_NOT_ABANDONED",
    "test_evaluation": "LOCKED_NOT_RUN",
    "final_method": "vanilla KD (Task 4 control)",
    "final_validation_mean": 0.9512,
    "final_validation_std": 0.00151,
    "notebook": "ATDL_post_task4_end_to_end_research.ipynb",
    "methods_executed": ["Task 3", "vanilla Task 4", "DKD confirmation", "DIST smoke", "DIST screen"],
    "methods_skipped": {"new_training": "No remaining implemented method has evidence above the completed Task 4 control."},
    "future_resumption_point": "Review and authorize one new supported one-seed method; preserve the test firewall and append artifacts.",
})
STATE_PATH.write_text(json.dumps(checkpoint, indent=2) + "\n")
print(json.dumps({k: checkpoint[k] for k in ("run_mode", "active_training", "final_method", "final_validation_mean", "test_evaluation", "automl_status")}, indent=2))

{
  "run_mode": "TIME-CONSTRAINED FINALIZATION",
  "active_training": null,
  "final_method": "vanilla KD (Task 4 control)",
  "final_validation_mean": 0.9512,
  "test_evaluation": "LOCKED_NOT_RUN",
  "automl_status": "PAUSED_NOT_ABANDONED"
}
